# Step Execution Time Table Generator

This notebook imports the helper functions from `generate_step_size_table.py` to generate and display publication-grade step execution time comparison tables across algorithms and workloads.

You can select which algorithms (`'sfadamw'`, `'muon'`, `'nadamw'`, `'ademamix'`, `'cautious_nadamw'`, `'lion'`, `'diloco'`, or `'all'`) to include in the table, and view both the Markdown and LaTeX source code directly in the notebook console.

In [1]:
import sys
from pathlib import Path

# Ensure the directory containing generate_step_size_table.py is in sys.path
script_dir = Path('./').resolve()
if str(script_dir) not in sys.path:
    sys.path.append(str(script_dir))

import generate_step_size_table
from generate_step_size_table import (
    ALGO_CONFIGS,
    resolve_selected_algorithms,
    get_selected_submissions,
    find_workloads,
    collect_step_times,
    format_table_values,
    generate_markdown_table,
    generate_latex_table,
    generate_tables
)

# Display available algorithms in registry
print("Available algorithms in registry:")
for algo_key, algo_info in ALGO_CONFIGS.items():
    versions = list(algo_info['submissions'].values())
    print(f"  - '{algo_key}' ({algo_info['display_name']}): {len(versions)} submissions -> {versions}")

Available algorithms in registry:
  - 'sfadamw' (Schedule-Free AdamW): 4 submissions -> ['JAX Schedule-Free v1', 'JAX Schedule-Free v2', 'PyTorch Schedule-Free v1', 'PyTorch Schedule-Free v2']
  - 'muon' (Muon): 5 submissions -> ['JAX Muon v1', 'PyTorch Muon v1', 'PyTorch Muon v2 (JAX HPS)', 'PyTorch Muon Replicated (JAX HPS)', 'PyTorch Muon Replicated (Torch HPS)']
  - 'nadamw' (NAdamW): 3 submissions -> ['JAX NAdamW v1', 'JAX NAdamW Baseline v0.5', 'JAX NAdamW ResNet']
  - 'ademamix' (AdemaMix): 1 submissions -> ['PyTorch AdemaMix']
  - 'cautious_nadamw' (Cautious NAdamW): 1 submissions -> ['JAX Cautious NAdamW']
  - 'lion' (Lion): 1 submissions -> ['PyTorch Lion']
  - 'diloco' (DiLoCo): 2 submissions -> ['PyTorch DiLoCo v1', 'PyTorch DiLoCo v2']


## Select Algorithms & Generate Tables

You can set `ALGO_NAMES` to:
- `'all'` to generate a single comprehensive table containing all algorithms, versions, and languages.
- A single algorithm like `'sfadamw'` or `'muon'`.
- A list of algorithms like `['sfadamw', 'muon']` or comma-separated string `'sfadamw,muon'` to compare multiple specific algorithms side by side.

All multiple versions and frameworks (JAX, PyTorch, v1/v2, etc.) for every selected algorithm will automatically be included in the table.

In [2]:
# User interactive variables - feel free to change these
ALGO_NAMES = 'all'                                     # Options: 'all', 'sfadamw', 'muon', ['sfadamw', 'muon'], etc.
LOG_DIR = '~/submissions_algorithms/logs/self_tuning'  # Base log directory containing study folders
SAVE_DIR = None                                        # Optional: set to path like './output_tables' to save markdown & latex files

# Generate the table across all selected algorithms, versions, and languages
output = generate_tables(
    algo_args=ALGO_NAMES,
    base_log_dir=LOG_DIR,
    save_dir=SAVE_DIR
)

# Display the Markdown Table
# print("=================== MARKDOWN TABLE ===================")
# print(output['markdown_table'])

# Display the LaTeX Table
print("\n==================== LATEX TABLE =====================")
print(output['latex_table'])


==================== LATEX TABLE =====================
\begin{table*}[t]
\centering
\caption{Step Execution Time Comparison (Normalized Ratios relative to Schedule-Free AdamW v2) across different workloads.}
\label{tab:step_time_comparison}
\begin{tabular}{lrrrrrrrrr}
\toprule
Optimizer & criteo1tb & fastmri & finewebedu\_lm & imagenet\_resnet & imagenet\_vit & librispeech\_conformer & librispeech\_deepspeech & ogbg & wmt \\
\midrule
JAX Schedule-Free v1 & $1.67 \pm 0.45$ & $1.29 \pm 0.15$ & $1.00$ & $0.63 \pm 0.04$ & $1.21 \pm 0.36$ & $0.95 \pm 0.03$ & $0.46$ & $1.02 \pm 0.01$ & $1.00$ \\
JAX Schedule-Free v2 & $1.00$ & $1.00 \pm 0.05$ & $1.00$ & $1.00 \pm 0.09$ & $1.00 \pm 0.05$ & $1.00 \pm 0.02$ & $1.00$ & $1.00$ & $1.00$ \\
PyTorch Schedule-Free v1 & $1.39 \pm 0.42$ & $1.15 \pm 0.21$ & $0.99$ & $0.97$ & $0.91$ & $0.96$ & $0.86$ & $1.05$ & $1.06$ \\
PyTorch Schedule-Free v2 & $1.00 \pm 0.04$ & $1.00 \pm 0.09$ & $1.00$ & $1.00 \pm 0.10$ & $1.00 \pm 0.03$ & $1.00$ & $1.00 \pm 0.05$ &

## Fine-Grained Step-by-Step Customization (Optional)

If you need to inspect raw step times, filter specific workloads, or customize the table generation step-by-step, you can directly invoke the modular helper functions:

In [3]:
# 1. Resolve selected algorithms and gather submissions
selected_algos = resolve_selected_algorithms(['sfadamw', 'muon'])
submissions_map = get_selected_submissions(selected_algos)

# 2. Discover available workloads across these submissions
base_path = Path(LOG_DIR).expanduser()
workloads = find_workloads(base_path, submissions_map)
print(f"Workloads found: {workloads}\n")

# 3. Collect step times and compute raw/formatted statistics
results, raw_table = collect_step_times(base_path, submissions_map, workloads)
formatted_table = format_table_values(results, workloads)

# 4. Inspect raw mean step times (ms/step) for a specific workload
print("Raw mean step times for 'finewebedu_lm':")
for opt_display in [disp for disp, _ in submissions_map.values()]:
    raw_val = raw_table[opt_display]['finewebedu_lm']
    if raw_val is not None:
        print(f"  {opt_display}: {raw_val:.2f} ms")

Workloads found: ['criteo1tb', 'fastmri', 'finewebedu_lm', 'imagenet_resnet', 'imagenet_vit', 'librispeech_conformer', 'librispeech_deepspeech', 'ogbg', 'wmt']

Raw mean step times for 'finewebedu_lm':
  JAX Schedule-Free v1: 224.69 ms
  JAX Schedule-Free v2: 225.71 ms
  PyTorch Schedule-Free v1: 390.75 ms
  PyTorch Schedule-Free v2: 395.64 ms
  JAX Muon v1: 505.06 ms
  PyTorch Muon v1: 392.11 ms
  PyTorch Muon v2 (JAX HPS): 390.16 ms
  PyTorch Muon Replicated (JAX HPS): 421.15 ms
  PyTorch Muon Replicated (Torch HPS): 427.71 ms
